In [71]:
#  echo: false
# Recharge automatiquement les modules modifiés (évite de redémarrer le kernel)
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as sci
from scipy.signal import hilbert
from scipy.stats import kurtosis
from scipy.fftpack import fft, fftfreq
from IPython.display import Markdown

# Cherche le dossier racine du projet (BDS_Fault_Detection) pour faire les imports depuis Sources/
dossier = Path().resolve()
for parent in (dossier, *dossier.parents):
    if "BDS_Fault_Detection" in parent.name:
        RACINE = parent
sys.path.insert(0, str(RACINE))

# Importation des fonctions du projet (dossier Sources/). Attention de l'ordre d'importation  !
from Sources import data, config, signaux, indicateurs, features 


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [72]:
catalogue = data.catalogue          # DataFrame indexé par nom lisible
charger = data.charger              # Fonction de chargement de données

In [73]:
# Sélection du dataset
selection = data.selectionner(sources=("12kDE","12kFE",),
                                  defauts=("IR", "B", "OR"),
                                  diametres=(7, 14, 21),
                                  positions_or=("6",),
                                  charges=(0, 1, 2, 3),
                                  avec_sains=True)
# selection.groupby(["defaut", "diametre_mils"]).size().unstack(fill_value=0)

In [74]:
selection

,numero,source,fs,defaut,position_OR,diametre_mils,charge_hp,rpm_nominal
nom,,,,,,,,
Normal_L0,97,Normal,12000,Sain,NaN,0,0,1797
Normal_L1,98,Normal,12000,Sain,NaN,0,1,1772
Normal_L2,99,Normal,12000,Sain,NaN,0,2,1750
Normal_L3,100,Normal,12000,Sain,NaN,0,3,1730
12kDE_IR007_L0,105,12kDE,12000,IR,NaN,7,0,1797
...,...,...,...,...,...,...,...,...
12kFE_B021_L0,290,12kFE,12000,B,NaN,21,0,1797
12kFE_B021_L1,291,12kFE,12000,B,NaN,21,1,1772
12kFE_B021_L2,292,12kFE,12000,B,NaN,21,2,1750


In [75]:
table = features.construire_table(selection, charger, canaux=("DE", "FE"))
features.sauvegarder_table(table)                # écrit data/features/table_indicateurs.csv

INDICATEURS = indicateurs.COLONNES_TEMPORELS   # 4 indicateurs temporels (partie exploratoire)
ORDRE = ["Sain", "IR007", "IR014", "IR021", "B007", "B014", "B021",
         "OR007", "OR014", "OR021"]

In [76]:
# table.iloc[70:100, :15]
table[table["canal"] == "FE"]

,fichier,fenetre,rpm,canal,numero,source,fs,defaut,position_OR,diametre_mils,...,amp_BSF_3,amp_FTF_1,amp_FTF_2,amp_FTF_3,ratio_BPFI,ratio_BPFO,ratio_BSF,ratio_FTF,f_pic,classe
101,Normal_L0,0,1796.0,FE,97,Normal,12000,Sain,NaN,0,...,0.001084,0.006267,0.004599,0.006184,0.108784,0.357421,0.249042,0.284753,60.058594,Sain
102,Normal_L0,1,1796.0,FE,97,Normal,12000,Sain,NaN,0,...,0.000480,0.004294,0.005294,0.006663,0.137428,0.467457,0.170645,0.224471,60.791016,Sain
103,Normal_L0,2,1796.0,FE,97,Normal,12000,Sain,NaN,0,...,0.000901,0.007760,0.006890,0.006910,0.122588,0.355864,0.092475,0.429073,60.058594,Sain
104,Normal_L0,3,1796.0,FE,97,Normal,12000,Sain,NaN,0,...,0.000896,0.003905,0.010554,0.013202,0.198674,0.512582,0.091343,0.197401,30.761719,Sain
105,Normal_L0,4,1796.0,FE,97,Normal,12000,Sain,NaN,0,...,0.001097,0.008527,0.008870,0.011672,0.193604,0.298059,0.157617,0.350719,31.494141,Sain
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8017,12kFE_OR021at6_L0,45,1797.0,FE,315,12kFE,12000,OR,6,21,...,0.063823,0.056750,0.586948,0.669555,0.109431,0.541788,0.284245,0.064536,30.029297,OR021
8018,12kFE_OR021at6_L0,46,1797.0,FE,315,12kFE,12000,OR,6,21,...,0.028864,0.029306,0.318542,0.330890,0.199736,0.660242,0.098169,0.041853,91.552734,OR021
8019,12kFE_OR021at6_L0,47,1797.0,FE,315,12kFE,12000,OR,6,21,...,0.022011,0.019165,0.560075,0.656948,0.174760,0.494200,0.310677,0.020363,30.029297,OR021
8020,12kFE_OR021at6_L0,48,1797.0,FE,315,12kFE,12000,OR,6,21,...,0.047168,0.030995,0.583986,0.674855,0.110190,0.549091,0.306058,0.034660,30.029297,OR021


In [80]:
table.groupby(["classe", "canal"]).size().unstack()


canal,DE,FE
classe,,
B007,401,401
B014,400,400
B021,400,400
IR007,401,401
IR014,400,400
IR021,400,400
OR007,402,402
OR014,250,250
OR021,251,251


In [78]:
#| label: preparation-modeling
#| code-summary: "Split par fichier + standardisation"
#| tbl-cap: "Répartition des fichiers entre apprentissage et test."

X_train, X_test, y_train, y_test, g_train, g_test = features.preparer_modeling_strat(
    table, indicateurs.COLONNES_INDICATEURS)

pd.DataFrame({
    "split": ["train", "test"],
    "fichiers": [g_train.nunique(), g_test.nunique()],
    "fenêtres": [len(X_train), len(X_test)],
    "classes présentes": [", ".join(sorted(y_train.unique())),
                          ", ".join(sorted(y_test.unique()))],
})

,split,fichiers,fenêtres,classes présentes
0,train,52,5916,"B007, B014, B021, IR007, IR014, IR021, OR007, ..."
1,test,18,2106,"B007, B014, B021, IR007, IR014, IR021, OR007, ..."
